# Week 9: CCE Ablation Study

This notebook validates that entropy spikes are a meaningful retrieval trigger:

1. **Same Setup** - Uses the exact same codebase and benchmark as Week 8
2. **CCE-Spike Retrieval** - Retrieve when CCE > 3.0 using query + confused tokens
3. **Ablation Baselines** - Random, Fixed-Interval, Query-Only, No-Retrieval
4. **Statistical Analysis** - Bootstrap CIs for rigorous comparison

---

## Setup

In [1]:
# Cell 1: Install dependencies
!pip install -q transformers torch accelerate scipy scikit-learn pandas matplotlib seaborn sentence-transformers

In [2]:
# Cell 2: Create directory structure
import os
import shutil

# Create module structure
os.makedirs('/content/orchestrator/entropy', exist_ok=True)
os.makedirs('/content/orchestrator/retrieval', exist_ok=True)
os.makedirs('/content/orchestrator/generation', exist_ok=True)
os.makedirs('/content/orchestrator/evaluation', exist_ok=True)

# Create root __init__.py
with open('/content/orchestrator/__init__.py', 'w') as f:
    f.write('"""Orchestrator package."""\n')

print("Directory structure created")

Directory structure created


In [3]:
# Cell 3: Upload module files
from google.colab import files

def upload_to_dir(target_dir):
    """Upload files and move to target directory."""
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith('.py'):
            dest = f'{target_dir}/{filename}'
            shutil.move(filename, dest)
            print(f"  OK {filename}")
    return uploaded

print("="*60)
print("STEP 1/4: Upload ENTROPY module files")
print("="*60)
print("\nUpload these files from packages/python-orchestrator/orchestrator/entropy/:")
print("  - __init__.py")
print("  - token_classifier.py  <-- REQUIRED for HybridClassifier")
print("  - (other entropy files as needed)")
upload_to_dir('/content/orchestrator/entropy')

print("\n" + "="*60)
print("STEP 2/4: Upload RETRIEVAL module files (4 files)")
print("="*60)
upload_to_dir('/content/orchestrator/retrieval')

print("\n" + "="*60)
print("STEP 3/4: Upload GENERATION module files (2 files)")
print("="*60)
upload_to_dir('/content/orchestrator/generation')

print("\n" + "="*60)
print("STEP 4/4: Upload EVALUATION module files (5 files)")
print("="*60)
print("\nUpload these files from packages/python-orchestrator/orchestrator/evaluation/:")
print("  - __init__.py")
print("  - benchmark.py")
print("  - benchmark_generator.py")
print("  - metrics.py")
print("  - runner.py")
upload_to_dir('/content/orchestrator/evaluation')

print("\n" + "="*60)
print("VERIFICATION")
print("="*60)
!ls -la /content/orchestrator/entropy/
!ls -la /content/orchestrator/evaluation/

STEP 1/4: Upload ENTROPY module files

Upload these files from packages/python-orchestrator/orchestrator/entropy/:
  - __init__.py
  - token_classifier.py  <-- REQUIRED for HybridClassifier
  - (other entropy files as needed)


Saving __init__.py to __init__.py
Saving calculator.py to calculator.py
Saving cce_computer.py to cce_computer.py
Saving measurement.py to measurement.py
Saving monitor.py to monitor.py
Saving spike_detector.py to spike_detector.py
Saving token_classifier.py to token_classifier.py
  OK __init__.py
  OK calculator.py
  OK cce_computer.py
  OK measurement.py
  OK monitor.py
  OK spike_detector.py
  OK token_classifier.py

STEP 2/4: Upload RETRIEVAL module files (4 files)


Saving __init__.py to __init__.py
Saving adaptive.py to adaptive.py
Saving context_manager.py to context_manager.py
Saving reposynth_retriever.py to reposynth_retriever.py
Saving topic_inference.py to topic_inference.py
  OK __init__.py
  OK adaptive.py
  OK context_manager.py
  OK reposynth_retriever.py
  OK topic_inference.py

STEP 3/4: Upload GENERATION module files (2 files)


Saving __init__.py to __init__.py
Saving adaptive_generator.py to adaptive_generator.py
  OK __init__.py
  OK adaptive_generator.py

STEP 4/4: Upload EVALUATION module files (5 files)

Upload these files from packages/python-orchestrator/orchestrator/evaluation/:
  - __init__.py
  - benchmark.py
  - benchmark_generator.py
  - metrics.py
  - runner.py


Saving __init__.py to __init__.py
Saving benchmark.py to benchmark.py
Saving benchmark_generator.py to benchmark_generator.py
Saving metrics.py to metrics.py
Saving runner.py to runner.py
Saving stats.py to stats.py
  OK __init__.py
  OK benchmark.py
  OK benchmark_generator.py
  OK metrics.py
  OK runner.py
  OK stats.py

VERIFICATION
total 120
drwxr-xr-x 2 root root  4096 Jan 25 06:22 .
drwxr-xr-x 6 root root  4096 Jan 25 06:22 ..
-rw-r--r-- 1 root root  9942 Jan 25 06:22 calculator.py
-rw-r--r-- 1 root root 10474 Jan 25 06:22 cce_computer.py
-rw-r--r-- 1 root root  6854 Jan 25 06:22 __init__.py
-rw-r--r-- 1 root root 11385 Jan 25 06:22 measurement.py
-rw-r--r-- 1 root root 23969 Jan 25 06:22 monitor.py
-rw-r--r-- 1 root root 17802 Jan 25 06:22 spike_detector.py
-rw-r--r-- 1 root root 21519 Jan 25 06:22 token_classifier.py
total 172
drwxr-xr-x 2 root root  4096 Jan 25 06:22 .
drwxr-xr-x 6 root root  4096 Jan 25 06:22 ..
-rw-r--r-- 1 root root 65176 Jan 25 06:22 benchmark_generator.py

In [4]:
# Cell 4: Base imports
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Any
from collections import defaultdict
import json
import time
import gc

sys.path.insert(0, '/content')

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

import warnings
warnings.filterwarnings('ignore')

print("Base imports complete")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Base imports complete
PyTorch: 2.9.0+cu126
CUDA available: True


In [5]:
# Cell 5: Import evaluation modules
from orchestrator.evaluation import (
    BenchmarkDataset,
    BenchmarkExample,
    EvaluationMetrics,
    EvaluationResult,
    create_benchmark,
    generate_full_benchmark,
    get_mock_codebase,
)
from orchestrator.evaluation.benchmark import (
    Difficulty,
    Category,
)
from orchestrator.evaluation.runner import (
    BaselineRunner,
    ExperimentRunner,
    BaselineMethod,
)

print("All evaluation modules imported successfully!")
print("\nAvailable components:")
print("  - BenchmarkDataset - structured evaluation examples")
print("  - EvaluationMetrics - 8 metrics for assessment")
print("  - generate_full_benchmark - create 100+ examples")
print("  - get_mock_codebase - Flask app codebase")

All evaluation modules imported successfully!

Available components:
  - BenchmarkDataset - structured evaluation examples
  - EvaluationMetrics - 8 metrics for assessment
  - generate_full_benchmark - create 100+ examples
  - get_mock_codebase - Flask app codebase


In [6]:
# Cell 10: Initialize metrics
metrics = EvaluationMetrics(embedding_model='all-MiniLM-L6-v2')
print("EvaluationMetrics initialized with improved hallucination detection")

EvaluationMetrics initialized with improved hallucination detection


In [7]:
# Cell 12: Embedding Retriever
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

class EmbeddingRetriever:
    def __init__(self, documents: Dict[str, str]):
        self.documents = documents
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.doc_names = list(documents.keys())
        self.doc_contents = list(documents.values())
        self.doc_embeddings = self.model.encode(self.doc_contents)
        print(f"EmbeddingRetriever: {len(documents)} documents indexed")

    def retrieve(self, query: str, top_k: int = 2, deduplicate: bool = True) -> List[Dict]:
        query_emb = self.model.encode([query])
        sims = cosine_similarity(query_emb, self.doc_embeddings)[0]
        top_idx = np.argsort(sims)[-top_k:][::-1]
        return [{'source': self.doc_names[i], 'content': self.doc_contents[i], 'score': float(sims[i])} for i in top_idx]


print("EmbeddingRetriever class defined")

EmbeddingRetriever class defined


In [8]:
# Cell 14: Token Classification & CCE Implementation (Using HybridClassifier)
from scipy.stats import entropy as scipy_entropy
from dataclasses import dataclass, field
from typing import Tuple
import random

# Import the advanced HybridClassifier from orchestrator
from orchestrator.entropy.token_classifier import HybridClassifier, KeywordClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

@dataclass
class MultiHopRetrievalResult:
    retrieved_files: List[str]
    retrieved_content: str
    scores: List[float]
    num_hops: int
    total_tokens: int
    trace: List[Dict[str, Any]]


class CCEQueryPlusTopKRetriever:
    """CCE Multi-Hop: Uses query + confused CODE tokens for retrieval.

    Uses HybridClassifier (keyword + embedding fallback) for token classification.
    """

    def __init__(self, base_retriever, tokenizer, model,
                 top_k: int = 2, max_retrievals: int = 5,
                 uncertainty_threshold: float = 3.0,
                 top_k_tokens: int = 10,
                 max_gen_tokens: int = 200,
                 cooldown_tokens: int = 5,
                 file_list_context: str = "",
                 use_hybrid_classifier: bool = True):
        self.retriever = base_retriever
        self.tokenizer = tokenizer
        self.model = model
        self.top_k = top_k
        self.max_retrievals = max_retrievals
        self.uncertainty_threshold = uncertainty_threshold
        self.top_k_tokens = top_k_tokens
        self.max_gen_tokens = max_gen_tokens
        self.cooldown_tokens = cooldown_tokens
        self.file_list_context = file_list_context
        self.use_hybrid_classifier = use_hybrid_classifier

        # Initialize classifier
        if use_hybrid_classifier:
            print("Using HybridClassifier (keyword + embedding fallback)")
            self.classifier = HybridClassifier(
                embedding_model='all-MiniLM-L6-v2',
                embedding_margin=0.05,
                use_embedding_cache=True
            )
        else:
            print("Using KeywordClassifier (keyword-only)")
            self.classifier = KeywordClassifier()

        # Build vocabulary classification (one-time)
        self._build_vocab_classification()

    def _build_vocab_classification(self):
        """Classify all tokens in vocabulary using HybridClassifier."""
        vocab_size = len(self.tokenizer)
        self.code_indices = []
        self.language_indices = []
        self.other_indices = []

        print(f"Classifying {vocab_size} tokens...")
        for token_id in range(vocab_size):
            try:
                token = self.tokenizer.decode([token_id]).strip()
                if not token:
                    self.other_indices.append(token_id)
                    continue

                # Use classifier (hybrid or keyword-only)
                result = self.classifier.classify(token)

                if result == 'code':
                    self.code_indices.append(token_id)
                elif result == 'language':
                    self.language_indices.append(token_id)
                else:
                    self.other_indices.append(token_id)
            except:
                self.other_indices.append(token_id)

        self.code_indices = np.array(self.code_indices)
        self.language_indices = np.array(self.language_indices)

        print(f"Vocab classification complete:")
        print(f"  Code tokens: {len(self.code_indices)}")
        print(f"  Language tokens: {len(self.language_indices)}")
        print(f"  Other tokens: {len(self.other_indices)}")

        # Show classifier stats if hybrid
        if self.use_hybrid_classifier and hasattr(self.classifier, 'get_stats'):
            stats = self.classifier.get_stats()
            total = stats['keyword_hits'] + stats['embedding_hits'] + stats['other']
            if total > 0:
                print(f"  Keyword hits: {stats['keyword_hits']} ({100*stats['keyword_hits']/total:.1f}%)")
                print(f"  Embedding hits: {stats['embedding_hits']} ({100*stats['embedding_hits']/total:.1f}%)")

    def _compute_cce(self, logits: torch.Tensor) -> Tuple[float, float, float]:
        logits_np = logits.cpu().numpy()

        if len(self.code_indices) > 0:
            code_logits = logits_np[self.code_indices]
            code_logits_stable = code_logits - np.max(code_logits)
            code_probs = np.exp(code_logits_stable) / np.sum(np.exp(code_logits_stable))
            h_code = float(scipy_entropy(code_probs, base=2))
        else:
            h_code = 0.0

        if len(self.language_indices) > 0:
            lang_logits = logits_np[self.language_indices]
            lang_logits_stable = lang_logits - np.max(lang_logits)
            lang_probs = np.exp(lang_logits_stable) / np.sum(np.exp(lang_logits_stable))
            h_lang = float(scipy_entropy(lang_probs, base=2))
        else:
            h_lang = 0.0

        return h_code - h_lang, h_code, h_lang

    def _extract_code_tokens_from_logits(self, logits: torch.Tensor) -> List[str]:
        logits_np = logits.cpu().numpy()
        code_logits = logits_np[self.code_indices]
        top_within_code = np.argsort(code_logits)[-self.top_k_tokens:][::-1]

        code_tokens = []
        for i in top_within_code:
            token_id = self.code_indices[i]
            token = self.tokenizer.decode([token_id]).strip()
            if len(token) > 1:
                code_tokens.append(token)
        return code_tokens

    def retrieve(self, query: str) -> MultiHopRetrievalResult:
        if self.file_list_context:
            prompt = f"{query}\n\n{self.file_list_context}\n\n"
        else:
            prompt = f"{query}\n\n"

        retrieved_files = []
        retrieved_content = []
        all_scores = []
        trace = []
        seen_files = set()
        retrieval_count = 0
        last_retrieval_pos = -100

        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)
        generated_ids = inputs['input_ids']

        for i in range(self.max_gen_tokens):
            with torch.no_grad():
                outputs = self.model(generated_ids)
                logits = outputs.logits[0, -1, :]

            del outputs  # Free memory
            cce, h_code, h_lang = self._compute_cce(logits)

            if i < 3:
                print(f"    Token {i}: CCE={cce:.3f} (H_code={h_code:.2f}, H_lang={h_lang:.2f})")

            tokens_since_last = i - last_retrieval_pos
            in_cooldown = tokens_since_last < self.cooldown_tokens

            if cce > self.uncertainty_threshold and not in_cooldown and retrieval_count < self.max_retrievals:
                confused_tokens = self._extract_code_tokens_from_logits(logits)
                retrieval_query = f"{query} {' '.join(confused_tokens)}"

                results = self.retriever.retrieve(retrieval_query, top_k=self.top_k, deduplicate=False)
                new_files = [r for r in results if r['source'] not in seen_files]

                if new_files:
                    for r in new_files:
                        seen_files.add(r['source'])
                        retrieved_files.append(r['source'])
                        retrieved_content.append(r['content'])
                        all_scores.append(r['score'])

                    new_context = "\n\n".join([r['content'] for r in new_files])
                    context_text = f"\n\nRelevant context:\n{new_context}\n\n"
                    context_ids = self.tokenizer.encode(context_text, return_tensors='pt').to(self.model.device)
                    generated_ids = torch.cat([generated_ids, context_ids], dim=-1)

                trace.append({
                    'hop': retrieval_count + 1,
                    'position': i,
                    'cce': cce,
                    'confused_tokens': confused_tokens[:5],
                    'new_files': [r['source'] for r in new_files] if new_files else [],
                })

                retrieval_count += 1
                last_retrieval_pos = i
                print(f"    SPIKE {retrieval_count} at {i}: CCE={cce:.2f}, tokens={confused_tokens[:5]}")

            next_token = torch.argmax(logits).unsqueeze(0).unsqueeze(0)
            generated_ids = torch.cat([generated_ids, next_token.to(self.model.device)], dim=-1)

            # Periodic memory cleanup
            if i % 50 == 0 and i > 0:
                torch.cuda.empty_cache()

            if next_token.item() == self.tokenizer.eos_token_id:
                break

        if not trace:
            trace.append({'hop': 0, 'method': 'no_spike_detected'})

        content = "\n\n".join(retrieved_content)
        print(f"    Total: {retrieval_count} retrievals, files: {retrieved_files}")

        return MultiHopRetrievalResult(
            retrieved_files=retrieved_files,
            retrieved_content=content,
            scores=all_scores,
            num_hops=retrieval_count,
            total_tokens=len(self.tokenizer.encode(content)) if content else 0,
            trace=trace
        )

print("CCEQueryPlusTopKRetriever defined (with HybridClassifier support)")

CCEQueryPlusTopKRetriever defined (with HybridClassifier support)


In [9]:
# Cell 13: Load LLM model (IMPROVED - Larger Model Options)

# === MODEL SELECTION ===
# Option 1: Qwen2.5-Coder-1.5B (recommended - 3x larger than 0.5B)
# Option 2: Qwen2.5-Coder-3B (if you have enough VRAM)
# Option 3: CodeLlama-7B-Instruct with 4-bit quantization

MODEL_OPTION = 1  # Change this to try different models

if MODEL_OPTION == 1:
    MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
    USE_QUANTIZATION = False
elif MODEL_OPTION == 2:
    MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
    USE_QUANTIZATION = False
elif MODEL_OPTION == 3:
    MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
    USE_QUANTIZATION = True  # 4-bit to fit in memory
else:
    MODEL_NAME = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
    USE_QUANTIZATION = False

print(f"Loading {MODEL_NAME}...")
print(f"Quantization: {USE_QUANTIZATION}")

if USE_QUANTIZATION:
    from transformers import BitsAndBytesConfig
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {MODEL_NAME}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")


Loading Qwen/Qwen2.5-Coder-1.5B-Instruct...
Quantization: False


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded: Qwen/Qwen2.5-Coder-1.5B-Instruct
Vocab size: 151665
Model parameters: 1,543,714,304


In [10]:
# Cell 15: Ablation Baselines (Memory-Optimized)

class RandomRetrievalBaseline:
    """Random retrieval baseline - uses upfront retrieval for efficiency."""
    def __init__(self, retriever, tokenizer, model, max_gen_tokens=100, file_list_context=""):
        self.retriever = retriever
        self.tokenizer = tokenizer
        self.model = model
        self.max_gen_tokens = max_gen_tokens
        self.file_list_context = file_list_context

    def retrieve(self, query: str, num_retrievals: int = 2, seed: int = None) -> MultiHopRetrievalResult:
        if seed is not None:
            random.seed(seed)

        results = self.retriever.retrieve(query, top_k=5)

        if len(results) > num_retrievals:
            selected = random.sample(results, num_retrievals)
        else:
            selected = results

        retrieved_files = [r['source'] for r in selected]
        retrieved_content = "\n\n".join([r['content'] for r in selected])
        scores = [r['score'] for r in selected]

        trace = [{'position': random.randint(10, self.max_gen_tokens-10), 'method': 'random'}
                 for _ in range(len(selected))]

        tokens_used = len(self.tokenizer.encode(retrieved_content)) if retrieved_content else 0

        return MultiHopRetrievalResult(
            retrieved_files, retrieved_content, scores,
            len(selected), tokens_used, trace
        )


class FixedIntervalBaseline:
    """Fixed interval baseline - uses upfront retrieval for efficiency."""
    def __init__(self, retriever, tokenizer, model, max_gen_tokens=100, file_list_context=""):
        self.retriever = retriever
        self.tokenizer = tokenizer
        self.model = model
        self.max_gen_tokens = max_gen_tokens
        self.file_list_context = file_list_context

    def retrieve(self, query: str, interval: int = 50, max_retrievals: int = 2) -> MultiHopRetrievalResult:
        results = self.retriever.retrieve(query, top_k=max_retrievals * 2)
        selected = results[:max_retrievals]

        retrieved_files = [r['source'] for r in selected]
        retrieved_content = "\n\n".join([r['content'] for r in selected])
        scores = [r['score'] for r in selected]

        trace = [{'position': (i+1) * interval, 'method': 'fixed_interval'}
                 for i in range(len(selected))]

        tokens_used = len(self.tokenizer.encode(retrieved_content)) if retrieved_content else 0

        return MultiHopRetrievalResult(
            retrieved_files, retrieved_content, scores,
            len(selected), tokens_used, trace
        )


class QueryOnlyBaseline:
    """Query-only baseline."""
    def __init__(self, retriever, tokenizer, model, max_gen_tokens=100):
        self.retriever = retriever
        self.tokenizer = tokenizer
        self.model = model
        self.max_gen_tokens = max_gen_tokens

    def generate(self, query: str) -> Tuple[str, List[str]]:
        results = self.retriever.retrieve(query, top_k=2)
        context = "\n\n".join([r['content'] for r in results])
        files = [r['source'] for r in results]

        prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
        inputs = self.tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                inputs['input_ids'],
                max_new_tokens=self.max_gen_tokens,
                pad_token_id=self.tokenizer.eos_token_id,
                do_sample=False
            )
        answer = self.tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

        del outputs, inputs
        torch.cuda.empty_cache()

        return answer, files


class NoRetrievalBaseline:
    """No retrieval baseline."""
    def __init__(self, tokenizer, model, max_gen_tokens=100):
        self.tokenizer = tokenizer
        self.model = model
        self.max_gen_tokens = max_gen_tokens

    def generate(self, query: str) -> str:
        prompt = f"Question: {query}\nAnswer:"
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                inputs['input_ids'],
                max_new_tokens=self.max_gen_tokens,
                pad_token_id=self.tokenizer.eos_token_id,
                do_sample=False
            )
        answer = self.tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

        del outputs, inputs
        torch.cuda.empty_cache()

        return answer

print("Memory-optimized baselines defined")


Memory-optimized baselines defined


In [11]:
# Cell 11: Define CCE Trace Function (uses masks from Cell 16)

# NOTE: This cell only DEFINES the function.
# The actual token masks (code_token_mask, lang_token_mask) are created in Cell 16
# when we initialize the CCE retriever.

def generate_with_cce_trace(query: str, max_tokens: int = 100) -> Dict:
    """Generate answer WITHOUT retrieval, logging CCE at each token position.

    Uses global code_token_mask and lang_token_mask created in Cell 16.
    """

    prompt = "Question: " + query + "\nAnswer:"
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(model.device)

    tokens_generated = []
    cce_trace = []
    h_code_trace = []
    h_lang_trace = []

    current_ids = input_ids

    for step in range(max_tokens):
        with torch.no_grad():
            outputs = model(current_ids)
            logits = outputs.logits[0, -1, :].float()

            # Verify shapes match
            vocab_size_model = logits.shape[0]

            probs = torch.softmax(logits, dim=-1).cpu().numpy()

            del outputs, logits

            # Use global masks (must be created in Cell 16 first!)
            # Safety check for size mismatch
            if len(probs) != len(code_token_mask):
                print(f"WARNING: Vocab mismatch! probs={len(probs)}, mask={len(code_token_mask)}")
                # Fallback: use simple entropy
                h_code = scipy_entropy(probs + 1e-10, base=2)
                h_lang = h_code * 0.7  # Rough estimate
            else:
                code_probs = probs[code_token_mask]
                lang_probs = probs[lang_token_mask]

                if code_probs.sum() > 1e-10:
                    code_probs_norm = code_probs / code_probs.sum()
                    h_code = scipy_entropy(code_probs_norm + 1e-10, base=2)
                else:
                    h_code = 0.0

                if lang_probs.sum() > 1e-10:
                    lang_probs_norm = lang_probs / lang_probs.sum()
                    h_lang = scipy_entropy(lang_probs_norm + 1e-10, base=2)
                else:
                    h_lang = 0.0

            cce = h_code - h_lang

            cce_trace.append(float(cce))
            h_code_trace.append(float(h_code))
            h_lang_trace.append(float(h_lang))

            next_token_id = int(np.argmax(probs))
            next_token = tokenizer.decode([next_token_id])
            tokens_generated.append(next_token)

            del probs

            if next_token_id == tokenizer.eos_token_id:
                break

            current_ids = torch.cat([current_ids, torch.tensor([[next_token_id]], device=model.device)], dim=1)

            if step % 25 == 0 and step > 0:
                gc.collect()
                torch.cuda.empty_cache()

    del current_ids, input_ids
    gc.collect()
    torch.cuda.empty_cache()

    # Use the SELECTED_THRESHOLD from Cell 16
    threshold = SELECTED_THRESHOLD if 'SELECTED_THRESHOLD' in dir() else 2.5

    return {
        'query': query,
        'answer': ''.join(tokens_generated),
        'tokens': tokens_generated,
        'cce_trace': cce_trace,
        'h_code_trace': h_code_trace,
        'h_lang_trace': h_lang_trace,
        'spike_positions': [i for i, cce in enumerate(cce_trace) if cce > threshold],
    }

print("generate_with_cce_trace function defined")
print("NOTE: Run Cell 16 first to create token masks before using this function!")


generate_with_cce_trace function defined
NOTE: Run Cell 16 first to create token masks before using this function!


In [12]:
# Cell 12: Compute hallucination trace function

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

hallu_embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def compute_hallucination_trace(tokens: List[str], ground_truth: str) -> List[Dict]:
    """
    Compute hallucination score at each token position using semantic distance.

    As generation progresses toward ground truth, similarity should INCREASE.
    A DROP in similarity indicates hallucination.
    """
    gt_embedding = hallu_embed_model.encode([ground_truth])[0]

    trace = []
    cumulative_text = ""
    prev_similarity = 0.0

    for i, token in enumerate(tokens):
        cumulative_text += token

        # Embed cumulative generated text
        gen_embedding = hallu_embed_model.encode([cumulative_text])[0]

        # Compute similarity to ground truth
        similarity = float(cosine_similarity([gen_embedding], [gt_embedding])[0][0])

        # Hallucination score = 1 - similarity (higher = more hallucinated)
        hallucination_score = 1.0 - similarity

        # Detect similarity drop (potential hallucination point)
        similarity_drop = prev_similarity - similarity if i > 0 else 0.0

        trace.append({
            'position': i,
            'token': token,
            'cumulative_similarity': similarity,
            'hallucination_score': hallucination_score,
            'similarity_drop': similarity_drop,
            'is_drop': similarity_drop > 0.01,
        })

        prev_similarity = similarity

    return trace

print("compute_hallucination_trace function defined")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

compute_hallucination_trace function defined


---
## Cerberus Experiment: Real Codebase Validation

Using **Cerberus** (Python validation library) as a real-world codebase to test CCE spike-error correlation.

In [13]:
# Cell A: Clone and Load Codebase (IMPROVED - More Novel Options)

import os
import subprocess

# === CODEBASE SELECTION ===
# Option 1: Cerberus (validation library - might be in training data)
# Option 2: httpx (modern async HTTP client - newer, less common)
# Option 3: Typer (CLI framework - newer)

CODEBASE_OPTION = 2  # Change this to try different codebases

CODEBASE_CONFIG = {
    1: {
        'name': 'cerberus',
        'url': 'https://github.com/pyeve/cerberus.git',
        'src_path': 'cerberus/cerberus',
        'description': 'Python validation library'
    },
    2: {
        'name': 'httpx',
        'url': 'https://github.com/encode/httpx.git',
        'src_path': 'httpx/httpx',
        'description': 'Modern async HTTP client (newer, less common)'
    },
    3: {
        'name': 'typer',
        'url': 'https://github.com/tiangolo/typer.git',
        'src_path': 'typer/typer',
        'description': 'CLI framework by FastAPI creator'
    }
}

config = CODEBASE_CONFIG[CODEBASE_OPTION]
REPO_NAME = config['name']
REPO_URL = config['url']
SRC_PATH = config['src_path']

print("="*70)
print(f"STEP 1: Clone {REPO_NAME.upper()} Repository")
print(f"Description: {config['description']}")
print("="*70)

# Clone if not exists
if not os.path.exists(REPO_NAME):
    subprocess.run(['git', 'clone', REPO_URL, '--depth', '1'], check=True)
    print(f"Cloned {REPO_NAME} repository")
else:
    print(f"{REPO_NAME} already exists, skipping clone")

# Load codebase into dict format
def load_codebase(src_path, repo_name):
    """Load source files into dict format for retriever."""
    files = {}

    for root, dirs, filenames in os.walk(src_path):
        # Skip test directories for cleaner results
        dirs[:] = [d for d in dirs if 'test' not in d.lower()]

        for f in filenames:
            if f.endswith('.py') and not f.startswith('test_'):
                path = os.path.join(root, f)
                rel_path = path.replace(f'{repo_name}/', '')
                try:
                    with open(path, 'r', encoding='utf-8') as fp:
                        content = fp.read()
                        if len(content) > 100:  # Skip tiny files
                            files[rel_path] = content
                except Exception as e:
                    print(f"  Warning: Could not read {path}: {e}")

    return files

target_codebase = load_codebase(SRC_PATH, REPO_NAME)

print(f"\n{REPO_NAME.upper()} Codebase Loaded")
print("="*70)
print(f"Total files: {len(target_codebase)}")
print(f"Total size: {sum(len(v) for v in target_codebase.values()):,} characters")
print("\nFiles:")
for path, content in sorted(target_codebase.items())[:15]:
    print(f"  {path} ({len(content):,} chars)")
if len(target_codebase) > 15:
    print(f"  ... and {len(target_codebase) - 15} more files")

# Create file list context
target_file_list = f"Available files in {REPO_NAME} codebase:\n"
for path in sorted(target_codebase.keys()):
    target_file_list += f"- {path}\n"
target_file_list += f"\nWhen answering questions about {REPO_NAME}, refer to the relevant files above."
print(f"\nFile list context created ({len(target_file_list)} chars)")


STEP 1: Clone HTTPX Repository
Description: Modern async HTTP client (newer, less common)
Cloned httpx repository

HTTPX Codebase Loaded
Total files: 23
Total size: 284,357 characters

Files:
  __init__.py (2,191 chars)
  __version__.py (108 chars)
  _api.py (11,743 chars)
  _auth.py (11,907 chars)
  _client.py (65,713 chars)
  _config.py (8,547 chars)
  _content.py (8,161 chars)
  _decoders.py (12,041 chars)
  _exceptions.py (8,490 chars)
  _main.py (15,626 chars)
  _models.py (44,697 chars)
  _multipart.py (9,843 chars)
  _status_codes.py (5,639 chars)
  _transports/__init__.py (275 chars)
  _transports/asgi.py (5,501 chars)
  ... and 8 more files

File list context created (480 chars)


In [14]:
# Cell B: Generate Benchmark Examples (IMPROVED - Dynamic based on codebase)

from dataclasses import dataclass
from typing import List
from collections import Counter

@dataclass
class BenchmarkExample:
    id: str
    category: str
    difficulty: str
    query: str
    ground_truth_files: List[str]
    keywords: List[str]
    ground_truth_answer: str

# Generate benchmark based on selected codebase
if CODEBASE_OPTION == 1:  # Cerberus
    BENCHMARK = [
        # API Usage
        {'id': 'api_001', 'category': 'api_usage', 'difficulty': 'easy',
         'query': 'How do I create a basic Cerberus validator and validate a document?',
         'ground_truth_files': ['cerberus/validator.py'], 'keywords': ['Validator', 'validate', 'schema'],
         'ground_truth_answer': 'Create a Validator instance with a schema dict, then call validate(document).'},
        {'id': 'api_002', 'category': 'api_usage', 'difficulty': 'medium',
         'query': 'How do I define a custom validation rule in Cerberus?',
         'ground_truth_files': ['cerberus/validator.py'], 'keywords': ['_validate', 'def', 'constraint'],
         'ground_truth_answer': 'Define a method named _validate_<rulename>(self, constraint, field, value).'},
        {'id': 'api_003', 'category': 'api_usage', 'difficulty': 'medium',
         'query': 'How do I validate nested documents in Cerberus?',
         'ground_truth_files': ['cerberus/validator.py'], 'keywords': ['schema', 'nested', 'dict'],
         'ground_truth_answer': "Use 'type': 'dict' with 'schema': {...} for nested documents."},
        # Implementation
        {'id': 'impl_001', 'category': 'implementation', 'difficulty': 'hard',
         'query': 'What method handles the normalization phase before validation?',
         'ground_truth_files': ['cerberus/validator.py'], 'keywords': ['normalize', '_normalize'],
         'ground_truth_answer': 'The _normalize method handles normalization before validation.'},
        {'id': 'impl_002', 'category': 'implementation', 'difficulty': 'hard',
         'query': 'How are schema rules discovered in Cerberus?',
         'ground_truth_files': ['cerberus/schema.py', 'cerberus/validator.py'], 'keywords': ['rules', '_validate_'],
         'ground_truth_answer': 'Schema rules are discovered via _validate_ method introspection.'},
        # Error handling
        {'id': 'err_001', 'category': 'error_handling', 'difficulty': 'medium',
         'query': 'What exception is raised when the schema itself is invalid?',
         'ground_truth_files': ['cerberus/errors.py'], 'keywords': ['SchemaError', 'invalid'],
         'ground_truth_answer': 'SchemaError is raised when the schema definition is invalid.'},
        {'id': 'err_002', 'category': 'error_handling', 'difficulty': 'hard',
         'query': 'What class handles error message formatting?',
         'ground_truth_files': ['cerberus/errors.py'], 'keywords': ['ErrorHandler', 'format'],
         'ground_truth_answer': 'BasicErrorHandler class handles error formatting.'},
    ]

elif CODEBASE_OPTION == 2:  # httpx
    BENCHMARK = [
        # API Usage
        {'id': 'api_001', 'category': 'api_usage', 'difficulty': 'easy',
         'query': 'How do I make a simple GET request with httpx?',
         'ground_truth_files': ['httpx/_api.py', 'httpx/_client.py'], 'keywords': ['get', 'request', 'Response'],
         'ground_truth_answer': 'Use httpx.get(url) for sync or async with httpx.AsyncClient().'},
        {'id': 'api_002', 'category': 'api_usage', 'difficulty': 'easy',
         'query': 'How do I send POST data with httpx?',
         'ground_truth_files': ['httpx/_api.py', 'httpx/_client.py'], 'keywords': ['post', 'data', 'json'],
         'ground_truth_answer': 'Use httpx.post(url, data=dict) or json=dict for JSON.'},
        {'id': 'api_003', 'category': 'api_usage', 'difficulty': 'medium',
         'query': 'How do I use httpx with async/await?',
         'ground_truth_files': ['httpx/_client.py'], 'keywords': ['AsyncClient', 'async', 'await'],
         'ground_truth_answer': 'Use async with httpx.AsyncClient() as client: await client.get(url).'},
        {'id': 'api_004', 'category': 'api_usage', 'difficulty': 'medium',
         'query': 'How do I set custom headers in httpx requests?',
         'ground_truth_files': ['httpx/_client.py'], 'keywords': ['headers', 'Headers'],
         'ground_truth_answer': 'Pass headers={"key": "value"} to request methods or Client constructor.'},
        {'id': 'api_005', 'category': 'api_usage', 'difficulty': 'medium',
         'query': 'How do I handle timeouts in httpx?',
         'ground_truth_files': ['httpx/_config.py', 'httpx/_client.py'], 'keywords': ['timeout', 'Timeout'],
         'ground_truth_answer': 'Use timeout=Timeout(connect=5.0, read=10.0) or timeout=30.0.'},
        # Implementation
        {'id': 'impl_001', 'category': 'implementation', 'difficulty': 'hard',
         'query': 'How does httpx handle HTTP/2 connections?',
         'ground_truth_files': ['httpx/_transports/default.py'], 'keywords': ['http2', 'HTTP2', 'transport'],
         'ground_truth_answer': 'httpx uses httpcore for transport, HTTP/2 enabled via http2=True.'},
        {'id': 'impl_002', 'category': 'implementation', 'difficulty': 'hard',
         'query': 'How does httpx handle connection pooling?',
         'ground_truth_files': ['httpx/_client.py', 'httpx/_transports/default.py'], 'keywords': ['pool', 'connection', 'limits'],
         'ground_truth_answer': 'Connection pooling via Limits class controlling max_connections.'},
        {'id': 'impl_003', 'category': 'implementation', 'difficulty': 'hard',
         'query': 'How does httpx encode request content?',
         'ground_truth_files': ['httpx/_content.py'], 'keywords': ['encode', 'content', 'stream'],
         'ground_truth_answer': 'Content encoding handled by encode_* functions in _content.py.'},
        # Error handling
        {'id': 'err_001', 'category': 'error_handling', 'difficulty': 'medium',
         'query': 'What exceptions can httpx raise for network errors?',
         'ground_truth_files': ['httpx/_exceptions.py'], 'keywords': ['HTTPError', 'NetworkError', 'TimeoutException'],
         'ground_truth_answer': 'HTTPError base class, NetworkError, TimeoutException, ConnectError.'},
        {'id': 'err_002', 'category': 'error_handling', 'difficulty': 'medium',
         'query': 'How do I handle HTTP status errors in httpx?',
         'ground_truth_files': ['httpx/_exceptions.py', 'httpx/_models.py'], 'keywords': ['raise_for_status', 'HTTPStatusError'],
         'ground_truth_answer': 'Call response.raise_for_status() to raise HTTPStatusError on 4xx/5xx.'},
        # Advanced
        {'id': 'adv_001', 'category': 'advanced', 'difficulty': 'hard',
         'query': 'How do I implement custom authentication in httpx?',
         'ground_truth_files': ['httpx/_auth.py'], 'keywords': ['Auth', 'authentication', 'flow'],
         'ground_truth_answer': 'Subclass Auth and implement __call__ or auth_flow generator.'},
        {'id': 'adv_002', 'category': 'advanced', 'difficulty': 'hard',
         'query': 'How do I create a custom transport in httpx?',
         'ground_truth_files': ['httpx/_transports/base.py'], 'keywords': ['BaseTransport', 'transport'],
         'ground_truth_answer': 'Subclass BaseTransport and implement handle_request method.'},
    ]

elif CODEBASE_OPTION == 3:  # Typer
    BENCHMARK = [
        # API Usage
        {'id': 'api_001', 'category': 'api_usage', 'difficulty': 'easy',
         'query': 'How do I create a basic Typer CLI application?',
         'ground_truth_files': ['typer/main.py'], 'keywords': ['Typer', 'app', 'command'],
         'ground_truth_answer': 'Create app = typer.Typer() and use @app.command() decorator.'},
        {'id': 'api_002', 'category': 'api_usage', 'difficulty': 'medium',
         'query': 'How do I add arguments to a Typer command?',
         'ground_truth_files': ['typer/main.py', 'typer/params.py'], 'keywords': ['Argument', 'Option'],
         'ground_truth_answer': 'Use function parameters with typer.Argument() or typer.Option().'},
        {'id': 'api_003', 'category': 'api_usage', 'difficulty': 'medium',
         'query': 'How do I create subcommands in Typer?',
         'ground_truth_files': ['typer/main.py'], 'keywords': ['add_typer', 'subcommand'],
         'ground_truth_answer': 'Use app.add_typer(sub_app, name="subcommand").'},
        # Implementation
        {'id': 'impl_001', 'category': 'implementation', 'difficulty': 'hard',
         'query': 'How does Typer convert Python type hints to CLI parameters?',
         'ground_truth_files': ['typer/main.py', 'typer/params.py'], 'keywords': ['type', 'annotation', 'click'],
         'ground_truth_answer': 'Typer inspects function annotations and maps to Click parameters.'},
        # Error handling
        {'id': 'err_001', 'category': 'error_handling', 'difficulty': 'medium',
         'query': 'How do I handle errors gracefully in Typer?',
         'ground_truth_files': ['typer/main.py'], 'keywords': ['Exit', 'Abort', 'exception'],
         'ground_truth_answer': 'Use typer.Exit(code=1) or typer.Abort() for controlled exits.'},
    ]

# Expand benchmark with variations
expanded_benchmark = []
for ex in BENCHMARK:
    expanded_benchmark.append(BenchmarkExample(**ex))

# Add more examples by creating variations
for ex in BENCHMARK[:5]:
    variation = ex.copy()
    variation['id'] = ex['id'] + '_v2'
    variation['query'] = 'Explain ' + ex['query'].lower().replace('how do i ', '').replace('?', '')
    expanded_benchmark.append(BenchmarkExample(**variation))

benchmark_examples = expanded_benchmark

print("="*70)
print(f"{REPO_NAME.upper()} BENCHMARK DATASET")
print("="*70)
print(f"Total examples: {len(benchmark_examples)}")
print(f"\nBy Category:")
cat_counts = Counter(ex.category for ex in benchmark_examples)
for cat, count in sorted(cat_counts.items()):
    print(f"  {cat}: {count}")
print(f"\nBy Difficulty:")
diff_counts = Counter(ex.difficulty for ex in benchmark_examples)
for diff, count in sorted(diff_counts.items()):
    print(f"  {diff}: {count}")


HTTPX BENCHMARK DATASET
Total examples: 17

By Category:
  advanced: 2
  api_usage: 10
  error_handling: 2
  implementation: 3

By Difficulty:
  easy: 4
  hard: 5
  medium: 8


In [15]:
# Cell C: Initialize Retrievers (Creates token masks for generate_with_cce_trace)

print("="*70)
print(f"INITIALIZING RETRIEVERS FOR {REPO_NAME.upper()}")
print("="*70)

# Create embedding retriever
target_retriever = EmbeddingRetriever(target_codebase)
print(f"Embedding retriever initialized with {len(target_codebase)} files")

# CCE Threshold
CCE_THRESHOLDS = [2.0, 2.5, 3.0, 3.5, 4.0]
SELECTED_THRESHOLD = 2.5

print(f"\nCCE Threshold: {SELECTED_THRESHOLD}")

# Create CCE retriever
print(f"Initializing CCE Retriever...")
target_cce_retriever = CCEQueryPlusTopKRetriever(
    base_retriever=target_retriever,
    tokenizer=tokenizer,
    model=model,
    uncertainty_threshold=SELECTED_THRESHOLD,
    max_gen_tokens=150,
    file_list_context=target_file_list
)
print("CCE retriever initialized!")

# === CREATE GLOBAL TOKEN MASKS ===
# These are used by generate_with_cce_trace in Cell 18
print("\nCreating global token masks...")

# Get vocab size directly from tokenizer (most reliable)
vocab_size = len(tokenizer)
print(f"Tokenizer vocab size: {vocab_size}")

# Verify with model config
model_vocab = model.config.vocab_size
print(f"Model config vocab size: {model_vocab}")

# Use the larger of the two to be safe
actual_vocab_size = max(vocab_size, model_vocab)
print(f"Using vocab size: {actual_vocab_size}")

# Create masks
code_token_mask = np.zeros(actual_vocab_size, dtype=bool)
lang_token_mask = np.zeros(actual_vocab_size, dtype=bool)

# Fill from CCE retriever's classification
for idx in target_cce_retriever.code_indices:
    if idx < actual_vocab_size:
        code_token_mask[idx] = True

for idx in target_cce_retriever.language_indices:
    if idx < actual_vocab_size:
        lang_token_mask[idx] = True

print(f"Token masks created: {code_token_mask.sum()} code, {lang_token_mask.sum()} language")

# Verify masks work
test_probs = np.random.rand(actual_vocab_size)
try:
    test_code = test_probs[code_token_mask]
    test_lang = test_probs[lang_token_mask]
    print(f"Mask verification: OK (code={len(test_code)}, lang={len(test_lang)})")
except Exception as e:
    print(f"Mask verification FAILED: {e}")

# Create baseline retrievers
target_random_baseline = RandomRetrievalBaseline(
    target_retriever, tokenizer, model, 150, target_file_list
)
target_fixed_baseline = FixedIntervalBaseline(
    target_retriever, tokenizer, model, 150, target_file_list
)

print("\nAll retrievers initialized!")

# Compute baseline tokens
sep = "\n\n"
target_full_context = sep.join([f"# {path}\n{content}" for path, content in list(target_codebase.items())[:10]])
target_baseline_tokens = len(tokenizer.encode(target_full_context[:30000]))
print(f"Baseline tokens (sample): {target_baseline_tokens:,}")


INITIALIZING RETRIEVERS FOR HTTPX
EmbeddingRetriever: 23 documents indexed
Embedding retriever initialized with 23 files

CCE Threshold: 2.5
Initializing CCE Retriever...
Using HybridClassifier (keyword + embedding fallback)
Classifying 151665 tokens...
Vocab classification complete:
  Code tokens: 43127
  Language tokens: 12952
  Other tokens: 95586
  Keyword hits: 2123 (1.4%)
  Embedding hits: 53956 (35.7%)
CCE retriever initialized!

Creating global token masks...
Tokenizer vocab size: 151665
Model config vocab size: 151936
Using vocab size: 151936
Token masks created: 43127 code, 12952 language
Mask verification: OK (code=43127, lang=12952)

All retrievers initialized!
Baseline tokens (sample): 6,827


In [16]:
# Cell D: Run Ablation Study (IMPROVED - More Examples + Threshold Analysis)

NUM_EXAMPLES = 20  # Increased from 15
BATCH_SIZE = 4

print("="*70)
print(f"{REPO_NAME.upper()} ABLATION STUDY (IMPROVED)")
print("="*70)
print(f"Examples: {NUM_EXAMPLES}")
print(f"CCE Threshold: {SELECTED_THRESHOLD}")
print(f"Model: {MODEL_NAME}")
print("="*70)

def cleanup_memory():
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def generate_answer(prompt: str, max_tokens: int = 120) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2000).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    del outputs, inputs
    cleanup_memory()
    return answer

# Sample examples
import random
random.seed(42)

if len(benchmark_examples) <= NUM_EXAMPLES:
    sampled_examples = benchmark_examples
else:
    # Stratified sampling by difficulty
    sampled_examples = []
    for diff in ['easy', 'medium', 'hard']:
        diff_examples = [ex for ex in benchmark_examples if ex.difficulty == diff]
        n_sample = min(len(diff_examples), NUM_EXAMPLES // 3 + 1)
        sampled_examples.extend(random.sample(diff_examples, n_sample))
    sampled_examples = sampled_examples[:NUM_EXAMPLES]

print(f"Sampled {len(sampled_examples)} examples")

# Storage
ablation_results = {
    'cce_spike': [],
    'random': [],
    'fixed': [],
    'query_only': [],
    'no_retrieval': []
}

# Track CCE behavior
cce_stats = {
    'total_spikes': 0,
    'total_retrievals': 0,
    'spike_positions': [],
    'cce_values': []
}

def evaluate_example(example, answer: str, retrieved_files: list, tokens_used: int,
                     method_name: str, num_retrievals: int = 0):
    return metrics.evaluate(
        example_id=example.id,
        generated_answer=answer,
        ground_truth_answer=example.ground_truth_answer,
        retrieved_files=retrieved_files,
        ground_truth_files=example.ground_truth_files,
        ground_truth_keywords=example.keywords,
        tokens_used=tokens_used,
        baseline_tokens=target_baseline_tokens,
        num_retrievals=num_retrievals,
        method=method_name
    )

# Process in batches
num_batches = (len(sampled_examples) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_idx in range(num_batches):
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(sampled_examples))
    batch = sampled_examples[start_idx:end_idx]

    print(f"\n--- Batch {batch_idx+1}/{num_batches} ---")

    for idx, ex in enumerate(batch):
        global_idx = start_idx + idx
        print(f"[{global_idx+1}/{len(sampled_examples)}] {ex.id}: {ex.query[:40]}...")

        # === CCE-Spike ===
        cleanup_memory()
        cce_result = target_cce_retriever.retrieve(ex.query)

        cce_stats['total_retrievals'] += cce_result.num_hops

        if cce_result.num_hops > 0 and cce_result.retrieved_content:
            prompt = f"Context:\n{cce_result.retrieved_content[:2500]}\n\nQuestion: {ex.query}\nAnswer:"
        else:
            prompt = f"Question: {ex.query}\nAnswer:"

        cce_answer = generate_answer(prompt)
        cce_eval = evaluate_example(ex, cce_answer, cce_result.retrieved_files,
                                    len(tokenizer.encode(prompt)), 'cce_spike', cce_result.num_hops)
        ablation_results['cce_spike'].append(cce_eval)
        print(f"  CCE: hops={cce_result.num_hops}, correct={cce_eval.answer_correctness:.3f}")

        # === Random ===
        cleanup_memory()
        rand_result = target_random_baseline.retrieve(ex.query, num_retrievals=max(2, cce_result.num_hops), seed=global_idx)
        if rand_result.retrieved_content:
            prompt = f"Context:\n{rand_result.retrieved_content[:2500]}\n\nQuestion: {ex.query}\nAnswer:"
        else:
            prompt = f"Question: {ex.query}\nAnswer:"

        rand_answer = generate_answer(prompt)
        rand_eval = evaluate_example(ex, rand_answer, rand_result.retrieved_files,
                                     len(tokenizer.encode(prompt)), 'random', rand_result.num_hops)
        ablation_results['random'].append(rand_eval)

        # === Fixed ===
        cleanup_memory()
        fixed_result = target_fixed_baseline.retrieve(ex.query, interval=40, max_retrievals=max(2, cce_result.num_hops))
        if fixed_result.retrieved_content:
            prompt = f"Context:\n{fixed_result.retrieved_content[:2500]}\n\nQuestion: {ex.query}\nAnswer:"
        else:
            prompt = f"Question: {ex.query}\nAnswer:"

        fixed_answer = generate_answer(prompt)
        fixed_eval = evaluate_example(ex, fixed_answer, fixed_result.retrieved_files,
                                      len(tokenizer.encode(prompt)), 'fixed', fixed_result.num_hops)
        ablation_results['fixed'].append(fixed_eval)

        # === Query-Only ===
        cleanup_memory()
        qo_results = target_retriever.retrieve(ex.query, top_k=3)
        qo_context = "\n\n".join([r['content'][:800] for r in qo_results])
        qo_prompt = f"Context:\n{qo_context}\n\nQuestion: {ex.query}\nAnswer:"
        qo_answer = generate_answer(qo_prompt)
        qo_eval = evaluate_example(ex, qo_answer, [r['source'] for r in qo_results],
                                   len(tokenizer.encode(qo_prompt)), 'query_only', 1)
        ablation_results['query_only'].append(qo_eval)

        # === No Retrieval ===
        cleanup_memory()
        nr_prompt = f"Question: {ex.query}\nAnswer:"
        nr_answer = generate_answer(nr_prompt)
        nr_eval = evaluate_example(ex, nr_answer, [], len(tokenizer.encode(nr_prompt)), 'no_retrieval', 0)
        ablation_results['no_retrieval'].append(nr_eval)

    print(f"  Batch complete.")
    cleanup_memory()
    import time
    time.sleep(0.5)

print("\n" + "="*70)
print("ABLATION STUDY COMPLETE")
print("="*70)
print(f"Total CCE retrievals: {cce_stats['total_retrievals']}")
print(f"Avg retrievals per example: {cce_stats['total_retrievals']/len(sampled_examples):.2f}")


HTTPX ABLATION STUDY (IMPROVED)
Examples: 20
CCE Threshold: 2.5
Model: Qwen/Qwen2.5-Coder-1.5B-Instruct
Sampled 17 examples

--- Batch 1/5 ---
[1/17] api_001: How do I make a simple GET request with ...
    Token 0: CCE=1.246 (H_code=4.09, H_lang=2.85)
    Token 1: CCE=-3.519 (H_code=0.44, H_lang=3.96)
    Token 2: CCE=4.005 (H_code=4.03, H_lang=0.02)
    SPIKE 1 at 2: CCE=4.00, tokens=['HTTP', 'this', 'GET', 'and', 'requests']
    SPIKE 2 at 7: CCE=3.25, tokens=['print', 'response', '```', 'with', 'async']
    SPIKE 3 at 20: CCE=3.69, tokens=['bin', 'ss', 'https', ':http', 'GET']
    SPIKE 4 at 40: CCE=3.73, tokens=['bin', 'https', 'https', 'http', ':http']
    SPIKE 5 at 55: CCE=3.50, tokens=['make', 'use', 'create', 'import', 'using']


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


    Total: 5 retrievals, files: ['__version__.py', '_transports/default.py', '__init__.py']
  CCE: hops=5, correct=0.655
[2/17] api_002: How do I send POST data with httpx?...
    Token 0: CCE=1.814 (H_code=4.69, H_lang=2.88)
    Token 1: CCE=-0.381 (H_code=3.99, H_lang=4.37)
    Token 2: CCE=-0.397 (H_code=0.16, H_lang=0.56)
    SPIKE 1 at 11: CCE=4.05, tokens=['it', 'Python', 'HTTP', 'this', 'as']
    SPIKE 2 at 27: CCE=4.40, tokens=['post', 'POST', 'post', '.Async', '.Session']
    SPIKE 3 at 32: CCE=6.23, tokens=['https', ':http', 'POST', 'https', 'post']
    SPIKE 4 at 52: CCE=5.58, tokens=['https', 'https', '/api', 'POST', 'endpoint']
    SPIKE 5 at 57: CCE=4.93, tokens=['/data', '.json', 'data', '-data', 'data']
    Total: 5 retrievals, files: ['__version__.py', '__init__.py', '_models.py', '_transports/default.py']
  CCE: hops=5, correct=0.731
[3/17] api_003: How do I use httpx with async/await?...
    Token 0: CCE=1.676 (H_code=4.79, H_lang=3.12)
    Token 1: CCE=-2.825 (H_cod

In [18]:
# Cell E: Spike-Error Correlation (with error handling)

from scipy.stats import pearsonr, chi2_contingency # Added this line

print("="*70)
print("SPIKE-ERROR CORRELATION EXPERIMENT")
print("="*70)

# Verify token masks exist and have correct size
print("\nVerifying token masks...")
try:
    vocab_from_model = model.config.vocab_size
    mask_size = len(code_token_mask)
    print(f"Model vocab: {vocab_from_model}, Mask size: {mask_size}")

    if mask_size < vocab_from_model:
        print("WARNING: Mask smaller than vocab! Extending...")
        code_token_mask_new = np.zeros(vocab_from_model, dtype=bool)
        lang_token_mask_new = np.zeros(vocab_from_model, dtype=bool)
        code_token_mask_new[:mask_size] = code_token_mask
        lang_token_mask_new[:mask_size] = lang_token_mask
        code_token_mask = code_token_mask_new
        lang_token_mask = lang_token_mask_new
        print(f"Extended masks to size {len(code_token_mask)}")
except Exception as e:
    print(f"Token mask check error: {e}")

# Use subset of examples
analysis_examples = sampled_examples[:min(10, len(sampled_examples))]
print(f"\nAnalyzing {len(analysis_examples)} examples")

spike_data = []

for idx, ex in enumerate(analysis_examples):
    print(f"\n[{idx+1}/{len(analysis_examples)}] {ex.id}...")

    try:
        cleanup_memory()
        trace_result = generate_with_cce_trace(ex.query, max_tokens=80)

        print(f"  Tokens: {len(trace_result['tokens'])}, Spikes: {len(trace_result['spike_positions'])}")

        spike_data.append({
            'example_id': ex.id,
            'query': ex.query,
            'ground_truth': ex.ground_truth_answer,
            'tokens': trace_result['tokens'],
            'cce_trace': trace_result['cce_trace'],
            'spike_positions': trace_result['spike_positions'],
        })
    except Exception as e:
        print(f"  ERROR: {e}")
        continue

    cleanup_memory()

print(f"\nCollected {len(spike_data)} traces")

if len(spike_data) < 3:
    print("\nNot enough data for correlation analysis!")
    spike_verdict = "INSUFFICIENT DATA"
    criteria_met = 0
    best_threshold = SELECTED_THRESHOLD
    final_r_pearson = 0
    final_relative_risk = 0
    final_p_chi2 = 1
else:
    # Compute hallucination traces
    print("\nComputing hallucination traces...")

    for data in spike_data:
        try:
            hallu_trace = compute_hallucination_trace(data['tokens'], data['ground_truth'])
            data['hallu_trace'] = hallu_trace
            data['hallu_scores'] = [h['hallucination_score'] for h in hallu_trace]
            data['drop_positions'] = [i for i, h in enumerate(hallu_trace) if h.get('is_drop', False)]
            print(f"  {data['example_id']}: {len(data['drop_positions'])} drops")
        except Exception as e:
            print(f"  {data['example_id']}: Error - {e}")
            data['hallu_scores'] = [0] * len(data['tokens'])
            data['drop_positions'] = []

    # Multi-threshold analysis
    print("\n" + "="*70)
    print("MULTI-THRESHOLD ANALYSIS")
    print("="*70)

    thresholds_to_test = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
    threshold_results = []

    for thresh in thresholds_to_test:
        all_cce = []
        all_hallu = []
        all_spike_flags = []
        all_drop_flags = []

        for data in spike_data:
            cce_trace = data['cce_trace']
            hallu_scores = data.get('hallu_scores', [0] * len(cce_trace))
            drop_positions = data.get('drop_positions', [])
            min_len = min(len(cce_trace), len(hallu_scores))

            for i in range(min_len):
                all_cce.append(cce_trace[i])
                all_hallu.append(hallu_scores[i])
                all_spike_flags.append(1 if cce_trace[i] > thresh else 0)
                all_drop_flags.append(1 if i in drop_positions else 0)

        if len(all_cce) < 10:
            continue

        all_cce = np.array(all_cce)
        all_hallu = np.array(all_hallu)
        all_spike_flags = np.array(all_spike_flags)
        all_drop_flags = np.array(all_drop_flags)

        n_spikes = sum(all_spike_flags)

        # Correlation
        if len(all_cce) > 2 and np.std(all_cce) > 0 and np.std(all_hallu) > 0:
            r_pearson, p_pearson = pearsonr(all_cce, all_hallu)
        else:
            r_pearson, p_pearson = 0, 1

        # Conditional probability
        spike_and_drop = sum((all_spike_flags == 1) & (all_drop_flags == 1))
        spike_no_drop = sum((all_spike_flags == 1) & (all_drop_flags == 0))
        no_spike_drop = sum((all_spike_flags == 0) & (all_drop_flags == 1))
        no_spike_no_drop = sum((all_spike_flags == 0) & (all_drop_flags == 0))

        total_spikes = spike_and_drop + spike_no_drop
        total_no_spikes = no_spike_drop + no_spike_no_drop

        p_drop_spike = spike_and_drop / total_spikes if total_spikes > 0 else 0
        p_drop_no_spike = no_spike_drop / total_no_spikes if total_no_spikes > 0 else 0

        relative_risk = p_drop_spike / p_drop_no_spike if p_drop_no_spike > 0 else 0

        # Chi-square
        contingency = np.array([[spike_and_drop, spike_no_drop], [no_spike_drop, no_spike_no_drop]])
        if contingency.min() > 0:
            try:
                chi2, p_chi2, _, _ = chi2_contingency(contingency)
            except:
                chi2, p_chi2 = 0, 1
        else:
            chi2, p_chi2 = 0, 1

        threshold_results.append({
            'threshold': thresh,
            'n_spikes': n_spikes,
            'n_positions': len(all_cce),
            'r_pearson': r_pearson,
            'relative_risk': relative_risk,
            'p_chi2': p_chi2,
        })

        print(f"\nThreshold {thresh}: Spikes={n_spikes}, r={r_pearson:.3f}, RR={relative_risk:.2f}x, p={p_chi2:.3f}")

    # Find best threshold
    if threshold_results:
        # Best = highest relative risk that's > 1
        valid_results = [r for r in threshold_results if r['relative_risk'] > 0]
        if valid_results:
            best_result = max(valid_results, key=lambda x: x['relative_risk'])
        else:
            best_result = threshold_results[0]

        print("\n" + "="*70)
        print("BEST THRESHOLD")
        print("="*70)
        print(f"Threshold: {best_result['threshold']}")
        print(f"Relative Risk: {best_result['relative_risk']:.2f}x")
        print(f"Pearson r: {best_result['r_pearson']:.4f}")

        # Final verdict
        print("\n" + "="*70)
        print("FINAL VERDICT")
        print("="*70)

        criteria_met = 0
        if abs(best_result['r_pearson']) > 0.15:
            print(f"[PASS] |r|={abs(best_result['r_pearson']):.3f} > 0.15")
            criteria_met += 1
        else:
            print(f"[FAIL] |r|={abs(best_result['r_pearson']):.3f} < 0.15")

        if best_result['relative_risk'] > 1.1:
            print(f"[PASS] RR={best_result['relative_risk']:.2f}x > 1.1x")
            criteria_met += 1
        else:
            print(f"[FAIL] RR={best_result['relative_risk']:.2f}x < 1.1x")

        if best_result['p_chi2'] < 0.15:
            print(f"[PASS] p={best_result['p_chi2']:.3f} < 0.15")
            criteria_met += 1
        else:
            print(f"[FAIL] p={best_result['p_chi2']:.3f} >= 0.15")

        print(f"\nCriteria met: {criteria_met}/3")

        if criteria_met >= 2:
            spike_verdict = "SPIKES PREDICT ERRORS"
        elif criteria_met == 1:
            spike_verdict = "WEAK EVIDENCE"
        else:
            spike_verdict = "NO EVIDENCE"

        print(f"VERDICT: {spike_verdict}")

        best_threshold = best_result['threshold']
        final_r_pearson = best_result['r_pearson']
        final_relative_risk = best_result['relative_risk']
        final_p_chi2 = best_result['p_chi2']
    else:
        print("No valid threshold results!")
        spike_verdict = "ANALYSIS FAILED"
        criteria_met = 0
        best_threshold = SELECTED_THRESHOLD
        final_r_pearson = 0
        final_relative_risk = 0
        final_p_chi2 = 1


SPIKE-ERROR CORRELATION EXPERIMENT

Verifying token masks...
Model vocab: 151936, Mask size: 151936

Analyzing 10 examples

[1/10] api_001...
  Tokens: 80, Spikes: 12

[2/10] api_002...
  Tokens: 80, Spikes: 12

[3/10] api_003...
  Tokens: 80, Spikes: 8

[4/10] api_004...
  Tokens: 80, Spikes: 10

[5/10] api_005...
  Tokens: 80, Spikes: 13

[6/10] impl_001...
  Tokens: 80, Spikes: 7

[7/10] impl_002...
  Tokens: 80, Spikes: 6

[8/10] impl_003...
  Tokens: 80, Spikes: 7

[9/10] err_001...
  Tokens: 80, Spikes: 7

[10/10] err_002...
  Tokens: 80, Spikes: 8

Collected 10 traces

Computing hallucination traces...
  api_001: 7 drops
  api_002: 2 drops
  api_003: 11 drops
  api_004: 6 drops
  api_005: 9 drops
  impl_001: 9 drops
  impl_002: 14 drops
  impl_003: 12 drops
  err_001: 12 drops
  err_002: 9 drops

MULTI-THRESHOLD ANALYSIS

Threshold 1.5: Spikes=154, r=0.052, RR=1.26x, p=0.400

Threshold 2.0: Spikes=129, r=0.052, RR=1.28x, p=0.392

Threshold 2.5: Spikes=90, r=0.052, RR=1.43x, p=0.

In [20]:
# Cell F: Export Results (IMPROVED)

print("="*70)
print(f"{REPO_NAME.upper()} EXPERIMENT RESULTS")
print("="*70)

# Compute aggregate metrics
methods = ['cce_spike', 'random', 'fixed', 'query_only', 'no_retrieval']
agg_results = {}

for method in methods:
    results_list = ablation_results[method]
    if results_list:
        agg_results[method] = {
            'correctness': float(np.mean([r.answer_correctness for r in results_list])),
            'hallucination': float(np.mean([r.hallucination_rate for r in results_list])),
            'composite': float(np.mean([r.get_composite_score() for r in results_list])),
            'n': len(results_list)
        }

print("")
print("=== ABLATION RESULTS ===")
print(f"{'Method':<15} {'Correct':>10} {'Hallu':>10} {'Composite':>10}")
print("-" * 50)
for method in methods:
    if method in agg_results:
        m = agg_results[method]
        print(f"{method:<15} {m['correctness']:>10.3f} {m['hallucination']:>10.3f} {m['composite']:>10.3f}")

# Find best method
best_method = max(agg_results.keys(), key=lambda x: agg_results[x]['composite'])
print(f"\nBest method: {best_method} (composite={agg_results[best_method]['composite']:.3f})")

# CCE vs baselines
cce_comp = agg_results['cce_spike']['composite']
rand_comp = agg_results['random']['composite']
fixed_comp = agg_results['fixed']['composite']

print("")
print("=== CCE vs BASELINES ===")
print(f"CCE Composite:    {cce_comp:.3f}")
print(f"Random Composite: {rand_comp:.3f}")
print(f"Fixed Composite:  {fixed_comp:.3f}")
print(f"CCE - Random:     {cce_comp - rand_comp:+.3f}")
print(f"CCE - Fixed:      {cce_comp - fixed_comp:+.3f}")

if cce_comp > rand_comp + 0.03:
    ablation_verdict = "CCE OUTPERFORMS RANDOM"
elif cce_comp < rand_comp - 0.03:
    ablation_verdict = "RANDOM OUTPERFORMS CCE"
else:
    ablation_verdict = "NO SIGNIFICANT DIFFERENCE"

print(f"Verdict: {ablation_verdict}")

print("")
print("=== SPIKE-ERROR CORRELATION ===")
print(f"Best threshold:  {best_threshold}")
print(f"Pearson r:       {final_r_pearson:.4f}")
print(f"Relative Risk:   {final_relative_risk:.2f}x")
print(f"Chi-square p:    {final_p_chi2:.4f}")
print(f"Criteria met:    {criteria_met}/3")
print(f"Verdict:         {spike_verdict}")

# Custom encoder for NumPy types
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)

# Export
export_data = {
    'experiment': f'{REPO_NAME}_cce_ablation_v2',
    'config': {
        'model': MODEL_NAME,
        'codebase': REPO_NAME,
        'cce_threshold': SELECTED_THRESHOLD,
        'best_threshold': best_threshold,
        'num_examples': len(sampled_examples),
    },
    'ablation': agg_results,
    'threshold_analysis': threshold_results,
    'spike_error': {
        'best_threshold': best_threshold,
        'pearson_r': float(final_r_pearson),
        'relative_risk': float(final_relative_risk) if final_relative_risk != float('inf') else 999,
        'chi2_p': float(final_p_chi2),
        'criteria_met': criteria_met,
        'verdict': spike_verdict,
    },
    'overall_verdict': ablation_verdict,
}

output_file = f'{REPO_NAME}_results_v2.json'
with open(output_file, 'w') as f:
    json.dump(export_data, f, indent=2, cls=NpEncoder)

print("")
print(f"Exported: {output_file}")
print("")
print("="*70)
print("EXPERIMENT COMPLETE")
print("="*70)

HTTPX EXPERIMENT RESULTS

=== ABLATION RESULTS ===
Method             Correct      Hallu  Composite
--------------------------------------------------
cce_spike            0.682      0.152      0.611
random               0.674      0.154      0.601
fixed                0.685      0.161      0.603
query_only           0.638      0.298      0.569
no_retrieval         0.662      0.147      0.577

Best method: cce_spike (composite=0.611)

=== CCE vs BASELINES ===
CCE Composite:    0.611
Random Composite: 0.601
Fixed Composite:  0.603
CCE - Random:     +0.010
CCE - Fixed:      +0.007
Verdict: NO SIGNIFICANT DIFFERENCE

=== SPIKE-ERROR CORRELATION ===
Best threshold:  3.5
Pearson r:       0.0515
Relative Risk:   1.93x
Chi-square p:    0.0748
Criteria met:    2/3
Verdict:         SPIKES PREDICT ERRORS

Exported: httpx_results_v2.json

EXPERIMENT COMPLETE
